In [149]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [150]:
df=pd.read_csv('Churn_Modelling.csv')

In [151]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [152]:
df.info()  # No need of RowNumber and CustomerID 

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.2 MB


In [153]:
df.nunique()   # Observation :- Gender can be label encoded  and   Georgraphy can be OneHotEncoded

RowNumber          10000
CustomerId         10000
Surname             2932
CreditScore          460
Geography              3
Gender                 2
Age                   70
Tenure                11
Balance             6382
NumOfProducts          4
HasCrCard              2
IsActiveMember         2
EstimatedSalary     9999
Exited                 2
dtype: int64

In [154]:
df.isnull().sum()  # No missing values

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [155]:
 # Some Libraries which coudl be used
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle

In [156]:
df.drop(columns=['RowNumber','CustomerId','Surname'],inplace=True)

In [157]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [158]:
le=LabelEncoder()
df['Gender']=le.fit_transform(df['Gender'])    # Label Encoding the Gender Column
ohe=OneHotEncoder()  
encoded_geo=ohe.fit_transform(df[['Geography']])
encoded_geo_column=pd.DataFrame(encoded_geo.toarray(),columns=ohe.get_feature_names_out(['Geography']),index=df.index)

In [159]:
df.drop(['Geography'],axis=1,inplace=True)
df=pd.concat([df,encoded_geo_column],axis=1)

In [160]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [161]:
# save the encoder and scaler in pikle format
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(le,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(ohe,file)

In [162]:
 # Now dividing the data in independent and dependent features
X=df.drop('Exited',axis=1)  # Independent / Input Features
y=df['Exited'] # Dependent / Output Features

In [163]:
X.shape,y.shape

((10000, 12), (10000,))

In [165]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)

# Scaling down all the features 
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [166]:
# saving the scaler file in pickle format
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [168]:
X.shape

(10000, 12)

# ANN Implementation

In [174]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime


In [176]:
# Building the model now

model=Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64,activation='relu'), # HL1 with 64 neurons and Relu activation function with 12 input functions from the input data
    Dense(32,activation='relu'), # HL2
    Dense(1,activation='sigmoid') # Output Layer
])

In [177]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# This can also be used for configuring the model

# import tensorflow
# opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
# loss_my=tensorflow.keras.losses.binary_crossentropy

In [180]:
 # Now we will compile(configure) the model
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
# Setup the Tensorboard for visualization
from tensorflow.keras.callbacks import TensorBoard,EarlyStopping

log_dir='logs/fit/' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [190]:
# Setup for Early stopping  (When the loss function is not changing after some epochs, then we stop training)

earlystopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [188]:
history=model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=100,callbacks=[tensorflow_callback,earlystopping_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8665 - loss: 0.3126 - val_accuracy: 0.8645 - val_loss: 0.3380
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8712 - loss: 0.3125 - val_accuracy: 0.8610 - val_loss: 0.3462
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8692 - loss: 0.3106 - val_accuracy: 0.8560 - val_loss: 0.3497
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8694 - loss: 0.3113 - val_accuracy: 0.8650 - val_loss: 0.3402
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8706 - loss: 0.3077 - val_accuracy: 0.8600 - val_loss: 0.3448
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8706 - loss: 0.3082 - val_accuracy: 0.8550 - val_loss: 0.3477
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8706 - loss: 0.3060 - val_accuracy: 0.8600 - val_loss: 0.3449
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8714 - loss: 0.3058 - val_accu

In [191]:
model.save('model.h5')  # Saving the model as h5 file as it is the compatable format for keras

In [198]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [200]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 29508), started 0:01:46 ago. (Use '!kill 29508' to kill it.)